<a href="https://colab.research.google.com/github/edwardmoreno18/especializacion/blob/main/Redes_Neuronales_Basicas_NumPy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Redes Neuronales Básicas con Python y NumPy

## Objetivo

Implementar desde cero, utilizando únicamente Python y **NumPy**, un perceptrón, una red neuronal de una capa y una red neuronal multicapa (Vainilla), con el fin de comprender el procesamiento de datos, las operaciones matriciales, la actualización de pesos y la generación de predicciones.

> **Nota:** No se utilizan librerías especializadas de Deep Learning como TensorFlow, Keras o PyTorch.


## 1. Conceptos básicos

Una red neuronal artificial recibe datos de entrada, los combina con **pesos** y un **sesgo (bias)**, aplica una función de activación y genera una salida.

La operación fundamental de una neurona puede expresarse como:

\[
z = XW + b
\]

donde:
- `X` representa los datos de entrada.
- `W` representa los pesos.
- `b` representa el sesgo.
- `z` es la combinación lineal antes de la activación.

La **vectorización** permite realizar estas operaciones sobre matrices completas mediante NumPy, evitando recorrer manualmente cada dato con ciclos innecesarios.


In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

print("NumPy está listo.")
print("Versión:", np.__version__)


NumPy está listo.
Versión: 2.1.3


## 2. Datos predefinidos y operaciones con NumPy

Para las prácticas se utilizarán datos pequeños y predefinidos. Cada fila representa una observación y cada columna una característica.

En este ejemplo, las dos características pueden interpretarse como valores normalizados entre 0 y 1.


In [2]:
X = np.array([
    [0.8, 0.6],
    [0.9, 0.8],
    [0.4, 0.8],
    [0.2, 0.1]
], dtype=float)

y = np.array([1, 1, 1, 0])

print("X =")
print(X)
print("\ny =")
print(y)
print("\nForma de X:", X.shape)
print("Forma de y:", y.shape)


X =
[[0.8 0.6]
 [0.9 0.8]
 [0.4 0.8]
 [0.2 0.1]]

y =
[1 1 1 0]

Forma de X: (4, 2)
Forma de y: (4,)


# 3. Perceptrón básico

El perceptrón es uno de los modelos neuronales más sencillos. Calcula una combinación lineal de las entradas y utiliza una función escalón para convertir el resultado en una clase.

En esta sección se implementa el algoritmo **desde cero**, incluyendo la actualización de pesos.


In [3]:
def funcion_escalon(z):
    return np.where(z >= 0, 1, 0)


def perceptron(X, y, tasa_aprendizaje=0.1, epocas=20):
    # Inicialización de pesos y bias
    pesos = np.zeros(X.shape[1])
    bias = 0.0

    for epoca in range(epocas):
        errores = 0

        for xi, objetivo in zip(X, y):
            z = np.dot(xi, pesos) + bias
            prediccion = funcion_escalon(z)

            error = objetivo - prediccion

            # Regla de actualización del perceptrón
            pesos += tasa_aprendizaje * error * xi
            bias += tasa_aprendizaje * error

            if error != 0:
                errores += 1

        if errores == 0:
            print(f"Convergencia alcanzada en la época {epoca + 1}.")
            break

    return pesos, bias


pesos_p, bias_p = perceptron(X, y)

print("Pesos finales:", pesos_p)
print("Bias final:", bias_p)


Convergencia alcanzada en la época 4.
Pesos finales: [0.1  0.09]
Bias final: -0.1


In [4]:
z_p = X @ pesos_p + bias_p
predicciones_p = funcion_escalon(z_p)

print("Combinación lineal:")
print(z_p)

print("\nPredicciones:")
print(predicciones_p)

print("\nExactitud:")
print(np.mean(predicciones_p == y))


Combinación lineal:
[ 0.034  0.062  0.012 -0.071]

Predicciones:
[1 1 1 0]

Exactitud:
1.0


### Interpretación del perceptrón

El perceptrón transforma varias características de entrada en una decisión binaria. La expresión `X @ pesos` corresponde a una multiplicación matricial vectorizada; posteriormente se suma el `bias` y se aplica la función escalón.

Durante el entrenamiento, los pesos se modifican cuando la predicción no coincide con el valor esperado. De esta manera, el modelo busca una frontera que permita separar las clases.


# 4. Red neuronal de una capa

Una red neuronal de una sola capa puede tener varias neuronas de salida. En este ejemplo se utilizará una capa con **dos neuronas**, una para cada clase, y una función sigmoide.

La operación vectorizada de la capa es:

\[
Z = XW + b
\]

y posteriormente:

\[
A = \sigma(Z) = \frac{1}{1 + e^{-Z}}
\]

Para mantener la práctica introductoria y usar solamente NumPy, el entrenamiento se realizará mediante descenso del gradiente.


In [5]:
def sigmoid(z):
    # Se limita el rango para evitar overflow numérico en np.exp
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


def entrenar_red_una_capa(X, Y, tasa_aprendizaje=0.5, epocas=3000):
    n_muestras, n_entradas = X.shape
    n_salidas = Y.shape[1]

    rng = np.random.default_rng(0)
    W = rng.normal(0, 0.1, size=(n_entradas, n_salidas))
    b = np.zeros((1, n_salidas))

    historial_error = []

    for epoca in range(epocas):
        # Propagación hacia adelante
        Z = X @ W + b
        A = sigmoid(Z)

        # Error cuadrático medio
        error = A - Y
        perdida = np.mean(error ** 2)
        historial_error.append(perdida)

        # Derivada de sigmoid
        dZ = error * A * (1 - A)

        # Gradientes vectorizados
        dW = (X.T @ dZ) / n_muestras
        db = np.mean(dZ, axis=0, keepdims=True)

        # Actualización de parámetros
        W -= tasa_aprendizaje * dW
        b -= tasa_aprendizaje * db

    return W, b, historial_error


In [6]:
# Problema OR codificado para dos neuronas de salida:
# [0, 0] -> [0, 0]
# [0, 1] -> [1, 0]
# [1, 0] -> [1, 0]
# [1, 1] -> [1, 0]

X_una_capa = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

Y_una_capa = np.array([
    [0, 1],
    [1, 0],
    [1, 0],
    [1, 0]
], dtype=float)

W1, b1, historial_1capa = entrenar_red_una_capa(
    X_una_capa, Y_una_capa
)

print("Matriz de pesos W:")
print(W1)

print("\nBias:")
print(b1)


Matriz de pesos W:
[[ 5.0181 -5.0169]
 [ 5.0181 -5.0169]]

Bias:
[[-2.2487  2.2481]]


In [7]:
Z_una_capa = X_una_capa @ W1 + b1
A_una_capa = sigmoid(Z_una_capa)

# La clase corresponde a la salida con mayor valor
pred_clase = np.argmax(A_una_capa, axis=1)

print("Activaciones:")
print(A_una_capa)

print("\nClases predichas:")
print(pred_clase)


Activaciones:
[[0.0955 0.9045]
 [0.941  0.059 ]
 [0.941  0.059 ]
 [0.9996 0.0004]]

Clases predichas:
[1 0 0 0]


### Interpretación de la red de una capa

En esta arquitectura, todas las predicciones se calculan en una sola transformación matricial. La vectorización permite procesar las cuatro observaciones simultáneamente mediante `X @ W + b`.

La red de una capa resulta adecuada para problemas que pueden representarse mediante una transformación lineal de las entradas. Para problemas más complejos se requiere introducir capas ocultas.


# 5. Red neuronal multicapa (Vainilla)

Una red multicapa incorpora al menos una **capa oculta**. Esto permite aprender relaciones no lineales.

Para evidenciar esta capacidad se utilizará el problema **XOR**, que no puede resolverse correctamente con un único clasificador lineal.

La arquitectura será:

- Entrada: 2 neuronas.
- Capa oculta: 4 neuronas con función sigmoide.
- Capa de salida: 1 neurona con sigmoide.

Las operaciones de propagación hacia adelante serán:

\[
Z^{[1]} = XW^{[1]} + b^{[1]}
\]

\[
A^{[1]} = ReLU(Z^{[1]})
\]

\[
Z^{[2]} = A^{[1]}W^{[2]} + b^{[2]}
\]

\[
A^{[2]} = \sigma(Z^{[2]})
\]


In [8]:
def entrenar_mlp(X, y, neuronas_ocultas=4, tasa_aprendizaje=2.0, epocas=20000):
    rng = np.random.default_rng(1)

    # Inicialización de parámetros
    W1 = rng.uniform(-1, 1, size=(X.shape[1], neuronas_ocultas))
    b1 = np.zeros((1, neuronas_ocultas))

    W2 = rng.uniform(-1, 1, size=(neuronas_ocultas, 1))
    b2 = np.zeros((1, 1))

    historial_error = []

    for epoca in range(epocas):
        # -------- Propagación hacia adelante --------
        Z1 = X @ W1 + b1
        A1 = sigmoid(Z1)

        Z2 = A1 @ W2 + b2
        A2 = sigmoid(Z2)

        # Entropía cruzada binaria
        eps = 1e-8
        perdida = -np.mean(
            y * np.log(A2 + eps) +
            (1 - y) * np.log(1 - A2 + eps)
        )
        historial_error.append(perdida)

        # -------- Retropropagación --------
        dZ2 = A2 - y
        dW2 = (A1.T @ dZ2) / X.shape[0]
        db2 = np.mean(dZ2, axis=0, keepdims=True)

        dA1 = dZ2 @ W2.T
        dZ1 = dA1 * A1 * (1 - A1)
        dW1 = (X.T @ dZ1) / X.shape[0]
        db1 = np.mean(dZ1, axis=0, keepdims=True)

        # -------- Actualización de parámetros --------
        W2 -= tasa_aprendizaje * dW2
        b2 -= tasa_aprendizaje * db2
        W1 -= tasa_aprendizaje * dW1
        b1 -= tasa_aprendizaje * db1

    return W1, b1, W2, b2, historial_error


In [9]:
# Datos XOR
X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y_xor = np.array([
    [0],
    [1],
    [1],
    [0]
], dtype=float)

W1_mlp, b1_mlp, W2_mlp, b2_mlp, historial_mlp = entrenar_mlp(
    X_xor, y_xor
)

print("W1:")
print(W1_mlp)

print("\nb1:")
print(b1_mlp)

print("\nW2:")
print(W2_mlp)

print("\nb2:")
print(b2_mlp)


W1:
[[ 6.6698  8.3053 -4.4458  4.7087]
 [-8.537  -6.6392  2.6538  4.2452]]

b1:
[[-2.8666  3.1583 -1.2152 -0.9122]]

W2:
[[ 17.6161]
 [-14.6542]
 [  4.4194]
 [  5.3191]]

b2:
[[0.6935]]


In [10]:
# Propagación final para obtener predicciones
Z1 = X_xor @ W1_mlp + b1_mlp
A1 = sigmoid(Z1)

Z2 = A1 @ W2_mlp + b2_mlp
A2 = sigmoid(Z2)

predicciones_mlp = (A2 >= 0.5).astype(int)

print("Salida de la red:")
print(A2)

print("\nPredicciones:")
print(predicciones_mlp)

print("\nValores esperados:")
print(y_xor.astype(int))

print("\nExactitud:")
print(np.mean(predicciones_mlp == y_xor))


Salida de la red:
[[0.0001]
 [0.9999]
 [0.9998]
 [0.0003]]

Predicciones:
[[0]
 [1]
 [1]
 [0]]

Valores esperados:
[[0]
 [1]
 [1]
 [0]]

Exactitud:
1.0


## 6. Comparación de los modelos

| Modelo | Capas | Activación principal | Característica |
|---|---:|---|---|
| Perceptrón | 1 | Escalón | Clasificación binaria lineal |
| Red de una capa | 1 | Sigmoide | Varias salidas mediante operaciones matriciales |
| Red multicapa | 2 o más | Sigmoide + Sigmoide | Puede representar relaciones no lineales |

La principal diferencia de la red multicapa es la existencia de una capa oculta. Esta capa permite transformar las características originales y construir representaciones más complejas.


## 7. Importancia de la vectorización

La vectorización consiste en aplicar operaciones matemáticas sobre arreglos completos en lugar de procesar cada elemento mediante ciclos manuales.

Por ejemplo:

```python
Z = X @ W + b
```

permite calcular simultáneamente la combinación lineal de varias muestras. Esto facilita la programación de redes neuronales, hace el código más compacto y aprovecha las operaciones optimizadas de NumPy.


## 8. Conclusiones

Durante esta práctica comprendí cómo funcionan internamente algunos modelos básicos de redes neuronales sin utilizar librerías especializadas de Deep Learning. La implementación del perceptrón permitió entender la relación entre entradas, pesos, bias, función de activación y predicción. También pude observar que las operaciones matriciales con NumPy permiten procesar varios datos de manera simultánea mediante vectorización.

Además, la implementación de la red de una capa y de la red multicapa me permitió identificar la diferencia entre una transformación lineal y un modelo con capas ocultas. En la red multicapa pude comprobar cómo la combinación de ReLU, sigmoide, propagación hacia adelante y retropropagación permite resolver un problema no lineal como XOR. Esta práctica fortaleció mi comprensión de la lógica matemática que existe detrás de las redes neuronales artificiales.
